# 📊 Advertising Sales Prediction
## Multiple Linear Regression — Complete Step-by-Step Project

**Business Scenario:** A consumer goods company advertises through **TV, Radio, and Newspaper** and wants to understand how these channels influence **Sales** and predict sales for a planned budget.

### Project objectives
1. Load and examine the dataset.
2. Use TV, Radio, and Newspaper as input features.
3. Use Sales as the target.
4. Train a Multiple Linear Regression model.
5. Predict sales for unseen data.
6. Predict sales for **TV = 150, Radio = 20, Newspaper = 30**.
7. Evaluate prediction error.
8. Interpret model coefficients.
9. Visualize actual vs predicted sales.
10. Give one business and one technical recommendation.

> **Beginner note:** Run the cells from top to bottom. Each section explains what we are doing and why.


## 1️⃣ Import the required libraries

We use:
- **pandas** → load and work with the dataset
- **matplotlib** → create the graph
- **scikit-learn** → train and evaluate the regression model


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

print("Libraries imported successfully!")


## 2️⃣ Load the dataset

The notebook expects `Advertising.csv` to be in the same folder as this notebook.

`pd.read_csv()` reads the CSV file and stores it in a pandas DataFrame called `df`.


In [ ]:
df = pd.read_csv("Advertising.csv")

print(df.head())


## 3️⃣ Examine the dataset

The dataset contains:
- `TV` → TV advertising spend
- `Radio` → Radio advertising spend
- `Newspaper` → Newspaper advertising spend
- `Sales` → product sales

We also check the number of rows and columns.


In [ ]:
print("Columns:")
print(df.columns.tolist())

print("\nDataset shape:")
print(df.shape)

print("\nDataset information:")
df.info()


## 4️⃣ Check for missing values and basic statistics

Missing values can cause problems during model training, so we check them first.

`describe()` gives useful statistics such as mean, minimum, maximum, and standard deviation.


In [ ]:
print("Missing values:")
print(df.isnull().sum())

print("\nSummary statistics:")
print(df.describe())


## 5️⃣ Define input features (X) and target (y)

We want the model to learn:

**TV + Radio + Newspaper → Sales**

So:
- `X` contains the three advertising channels.
- `y` contains Sales.


In [ ]:
X = df[["TV", "Radio", "Newspaper"]]
y = df["Sales"]

print("X:")
print(X.head())

print("\ny:")
print(y.head())


## 6️⃣ Split the data into training and testing sets

We use:
- **80% training data** → the model learns from this.
- **20% testing data** → we check how well it predicts unseen data.

`random_state=42` makes the split reproducible.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)


## 7️⃣ Create the Multiple Linear Regression model

Multiple Linear Regression is suitable because we have **three input variables** and one continuous target.

The model learns an equation like:

**Sales = intercept + (TV coefficient × TV) + (Radio coefficient × Radio) + (Newspaper coefficient × Newspaper)**


In [ ]:
model = LinearRegression()

print("Linear Regression model created.")


## 8️⃣ Train the model

`fit()` is the learning step.

The model looks at the historical advertising budgets and their corresponding sales values and learns the best coefficients.


In [ ]:
model.fit(X_train, y_train)

print("Model trained successfully!")


## 9️⃣ Predict sales for unseen test data

The test data was not used for training.

`predict()` uses the learned relationship to estimate sales for those unseen records.


In [ ]:
y_pred = model.predict(X_test)

results = pd.DataFrame({
    "Actual Sales": y_test.values,
    "Predicted Sales": y_pred
})

print(results.head(10))


## 🔟 Predict sales for the planned advertising budget

The company plans:

| Channel | Budget |
|---|---:|
| TV | 150 |
| Radio | 20 |
| Newspaper | 30 |

We create one new row and send it to the trained model.


In [ ]:
new_budget = pd.DataFrame({
    "TV": [150],
    "Radio": [20],
    "Newspaper": [30]
})

planned_sales = model.predict(new_budget)[0]

print(f"Predicted Sales for TV=150, Radio=20, Newspaper=30: {planned_sales:.2f}")


## 1️⃣1️⃣ Evaluate the model

We use four common metrics:

- **MAE (Mean Absolute Error):** average size of prediction error.
- **MSE (Mean Squared Error):** squares errors, so large errors get more penalty.
- **RMSE:** square root of MSE; easier to interpret because it is in Sales units.
- **R²:** shows how much variation in Sales is explained by the model. Closer to 1 is generally better.

For this standard Advertising dataset, an R² around 0.90 is expected with this train/test setup.


In [ ]:
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = mse ** 0.5
r2 = r2_score(y_test, y_pred)

print("MODEL EVALUATION")
print("----------------")
print(f"MAE  : {mae:.4f}")
print(f"MSE  : {mse:.4f}")
print(f"RMSE : {rmse:.4f}")
print(f"R²   : {r2:.4f}")


## 1️⃣2️⃣ Interpret the coefficients

Each coefficient estimates the change in Sales for a one-unit increase in that advertising channel **while the other channels are kept constant**.

A larger positive coefficient means a stronger positive estimated relationship in this model.

> **Important:** Coefficient size should not be treated as a complete ROI measure because the channels may have different scales, costs, and business conditions.


In [ ]:
coefficients = pd.DataFrame({
    "Advertising Medium": X.columns,
    "Coefficient": model.coef_
})

print("Intercept:", model.intercept_)
print("\nCoefficients:")
print(coefficients)


## 1️⃣3️⃣ Find the strongest and least impact

We compare the learned coefficients.

For this dataset, the usual result is:
- **Radio → strongest positive coefficient**
- **Newspaper → least impact / approximately zero coefficient**

This means Radio has the strongest estimated positive relationship with Sales after accounting for the other two channels.


In [ ]:
strongest = coefficients.loc[coefficients["Coefficient"].idxmax()]
least = coefficients.loc[coefficients["Coefficient"].idxmin()]

print(f"Strongest impact: {strongest['Advertising Medium']} "
      f"({strongest['Coefficient']:.4f})")

print(f"Least impact: {least['Advertising Medium']} "
      f"({least['Coefficient']:.4f})")


## 1️⃣4️⃣ Visualize Actual Sales vs Predicted Sales

Each dot represents one test record.

If predictions are good, the points should stay reasonably close to the ideal diagonal relationship where:

**Predicted Sales = Actual Sales**


In [ ]:
plt.figure(figsize=(8, 5))

plt.scatter(y_test, y_pred, alpha=0.8)

# Ideal prediction line
min_value = min(y_test.min(), y_pred.min())
max_value = max(y_test.max(), y_pred.max())
plt.plot([min_value, max_value], [min_value, max_value], linestyle="--")

plt.xlabel("Actual Sales")
plt.ylabel("Predicted Sales")
plt.title("Actual Sales vs Predicted Sales")
plt.grid(True, alpha=0.3)
plt.show()


## 1️⃣5️⃣ Optional: Compare actual and predicted values

This table makes it easier to inspect individual predictions.


In [ ]:
results["Error"] = results["Actual Sales"] - results["Predicted Sales"]

print(results.head(15))


# 🧠 Final Analysis

### Business recommendation
The company should consider giving more attention to **Radio advertising**, because Radio has the strongest positive coefficient in the regression model. Before changing the entire budget, the company should also compare campaign cost and actual return on investment.

### Technical improvement
Try stronger regression approaches such as **Random Forest, Gradient Boosting, or XGBoost**, and use **cross-validation** to compare models using MAE, RMSE, and R².

### Assignment conclusion
The Multiple Linear Regression model learns the relationship between TV, Radio, Newspaper advertising spend and Sales. It can predict sales for unseen advertising budgets and provides coefficients that help identify the relative estimated influence of each channel.


# 🎤 Quick Viva Questions & Answers

**Q1. Why is this a regression problem?**  
Because Sales is a continuous numerical value.

**Q2. What are the input features?**  
TV, Radio, and Newspaper.

**Q3. What is the target variable?**  
Sales.

**Q4. Why do we split the data?**  
To train on historical data and test performance on unseen data.

**Q5. What does `fit()` do?**  
It trains the model and learns the coefficients.

**Q6. What does `predict()` do?**  
It uses the trained model to estimate Sales.

**Q7. Which medium has the strongest impact?**  
Radio, based on the learned coefficient.

**Q8. Which medium has the least impact?**  
Newspaper, based on the coefficient in this model.

**Q9. What does R² mean?**  
It tells us how much variation in Sales is explained by the model.

**Q10. What is the prediction for TV=150, Radio=20, Newspaper=30?**  
Approximately **17.5 Sales units** with the standard Advertising dataset and a typical train/test split.
